In [22]:
import pandas as pd
import numpy as np

# Cargar datasets originales
df_vuelos = pd.read_csv(
    '../data/raw/TraficoAereoMensual2022-2026.csv'
)

df_kpi = pd.read_csv(
    '../data/raw/kpi_oferta_demanda.csv'
)

print("Dimensiones iniciales Vuelos:", df_vuelos.shape)
print("Dimensiones iniciales KPI:", df_kpi.shape)

print("\nNulos en vuelos:")
print(
    df_vuelos[
        ['Ano', 'Mes', 'Operador', 'PASAJEROS']
    ].isnull().sum()
)

print("\nDuplicados en vuelos:", df_vuelos.duplicated().sum())
print("Duplicados en KPI:", df_kpi.duplicated().sum())

Dimensiones iniciales Vuelos: (24629, 26)
Dimensiones iniciales KPI: (3050, 9)

Nulos en vuelos:
Ano          0
Mes          0
Operador     0
PASAJEROS    0
dtype: int64

Duplicados en vuelos: 0
Duplicados en KPI: 0


In [23]:
# Eliminar duplicados exactos
df_vuelos = df_vuelos.drop_duplicates()
df_kpi = df_kpi.drop_duplicates()

# Mantener únicamente registros de pasajeros
df_vuelos_limpio = df_vuelos[
    df_vuelos['PASAJEROS'] > 0
].copy()

# Eliminar pasajeros nulos
df_vuelos_limpio = df_vuelos_limpio[
    df_vuelos_limpio['PASAJEROS'].notna()
].copy()

print("Vuelos después de la limpieza:", len(df_vuelos_limpio))
print("KPI después de la limpieza:", len(df_kpi))

Vuelos después de la limpieza: 19912
KPI después de la limpieza: 3050


In [24]:
# Crear llave temporal YYYYMM
df_vuelos_limpio['month_id'] = (
    df_vuelos_limpio['Ano'].astype(str)
    + df_vuelos_limpio['Mes'].astype(str).str.zfill(2)
).astype(int)

# Normalizar columnas de texto
df_vuelos_limpio['Grupo'] = (
    df_vuelos_limpio['Grupo']
    .astype(str)
    .str.strip()
    .str.upper()
)

df_vuelos_limpio['NAC'] = (
    df_vuelos_limpio['NAC']
    .astype(str)
    .str.strip()
    .str.upper()
)

df_kpi['aerolinea_oaci'] = (
    df_kpi['aerolinea_oaci']
    .astype(str)
    .str.strip()
    .str.upper()
)

df_kpi['inter_dom'] = (
    df_kpi['inter_dom']
    .astype(str)
    .str.strip()
    .str.upper()
)

# Homologar grupo de la fuente de vuelos con código OACI del KPI
mapa_grupo_oaci = {
    'LATAM': 'LAN',
    'SKY': 'SKU',
    'JETSMART': 'JAT',
    'DAP': 'DAP'
}

df_vuelos_limpio['aerolinea_oaci'] = (
    df_vuelos_limpio['Grupo'].map(mapa_grupo_oaci)
)

# Homologar nacional/internacional con el formato del KPI
df_vuelos_limpio['inter_dom'] = (
    df_vuelos_limpio['NAC'].map({
        'NACIONAL': 'D',
        'INTERNACIONAL': 'I'
    })
)

print("Códigos sin equivalencia:")
print(
    sorted(
        df_vuelos_limpio.loc[
            df_vuelos_limpio['aerolinea_oaci'].isna(),
            'Grupo'
        ].unique()
    )
)

print("\nTipos de operación sin equivalencia:")
print(
    df_vuelos_limpio.loc[
        df_vuelos_limpio['inter_dom'].isna(),
        'NAC'
    ].unique()
)

Códigos sin equivalencia:
['AEROLINEAS ARGENTINAS', 'AEROMÉXICO', 'AIR CANADA', 'AIR FRANCE', 'AMASZONAS', 'AMERICAN AIRLINES', 'ARAJET', 'AVIANCA', 'BOLIVIANA DE AVIACIÓN', 'BRITISH AIRWAYS', 'CONVIASA', 'COPA', 'CUBANA DE AVIACION', 'DELTA AIRLINES', 'ESTELAR', 'GOL', 'IBERIA', 'KLM', 'KOREAN AIR', 'LEVEL', 'PARANAIR', 'QANTAS AIRWAYS', 'STAR UP', 'TURKISH AIRLINES', 'UNITED AIRLINES']

Tipos de operación sin equivalencia:
<ArrowStringArray>
[]
Length: 0, dtype: str


In [25]:
# Agregar tráfico al mismo nivel que el KPI
df_trafico_aerolinea = (
    df_vuelos_limpio
    .groupby(
        [
            'month_id',
            'aerolinea_oaci',
            'inter_dom'
        ],
        as_index=False
    )
    .agg(
        pasajeros=('PASAJEROS', 'sum'),
        rutas=('ORIG_1', 'nunique'),
        destinos=('DEST_1', 'nunique')
    )
)

# Una fila KPI por combinación de llaves
df_kpi_unico = df_kpi.drop_duplicates(
    subset=[
        'month_id',
        'aerolinea_oaci',
        'inter_dom'
    ]
)

# Merge entre tablas con el mismo nivel de agregación
df_aerolinea_kpi = pd.merge(
    df_trafico_aerolinea,
    df_kpi_unico,
    on=[
        'month_id',
        'aerolinea_oaci',
        'inter_dom'
    ],
    how='inner',
    validate='one_to_one'
)

print("Tráfico agregado:", df_trafico_aerolinea.shape)
print("Dataset final con KPI:", df_aerolinea_kpi.shape)

Tráfico agregado: (416, 6)
Dataset final con KPI: (416, 12)


In [26]:
# Dataset principal para rutas y pasajeros
df_vuelos_limpio.to_csv(
    '../data/processed/datos_limpios.csv',
    index=False
)

# Dataset agregado para análisis de KPI
df_aerolinea_kpi.to_csv(
    '../data/processed/datos_aerolinea_kpi.csv',
    index=False
)